In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── 1. Load the data ──────────────────────────────────────────────────────────
# If the file lives elsewhere, replace 'gene_expression.csv' with that path.
df = pd.read_csv('/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_25ct_20250619/comprehensive_summary_ism_90/motif_occurrence_by_celltype_combined_normalized.csv')

# Make the gene symbols the index (assumes first column holds them)
df = df.rename(columns={df.columns[0]: 'Gene'}).set_index('Gene')

# ── 2. Grab the top-N genes by total expression ───────────────────────────────
N = 100
top = df.sort_values('Total', ascending=False).head(N)

# Keep only the 25 cell-type columns (everything except 'Total')
expr = top.drop(columns='Total')

# ── 3. Plot ───────────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 24))
plt.imshow(expr, aspect='auto')             # default colormap → fine
plt.xticks(range(expr.shape[1]), expr.columns, rotation=90, ha='center')
plt.yticks(range(expr.shape[0]), expr.index)
plt.colorbar(label='Expression level')
plt.title(f'Top {N} genes by overall expression')
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram

# ── 1. Load and tidy ──────────────────────────────────────────────────────────
df = pd.read_csv('/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_25ct_20250619/comprehensive_summary_ism_90/motif_occurrence_by_celltype_combined_normalized.csv')
df = df.rename(columns={df.columns[0]: 'Gene'}).set_index('Gene')

expr = df.drop(columns='Total')         # keep only the 25 cell-type cols

# ── 2. Standardise (z-score) each gene so we cluster on patterns not scale ────
expr_z = expr.sub(expr.mean(axis=1), axis=0) \
             .div(expr.std(axis=1).replace(0, 1), axis=0)

# ── 3. Hierarchical clustering ────────────────────────────────────────────────
dist   = pdist(expr_z, metric='euclidean')            # pair-wise distances
linkage_mat = linkage(dist, method='average')         # UPGMA / average

# ── 4. Plot dendrogram + heat-map ─────────────────────────────────────────────
fig = plt.figure(figsize=(12, 8))

# 4a. Gene dendrogram (left)
ax_d = fig.add_axes([0.05, 0.1, 0.28, 0.8])           # [left, bottom, w, h]
dendro = dendrogram(linkage_mat,
                    orientation='left',
                    labels=expr_z.index,
                    leaf_font_size=8,
                    ax=ax_d)
ax_d.set_xlabel('Cluster distance')

# 4b. Re-order expression matrix to match dendrogram leaves
ordered_expr = expr_z.iloc[dendro['leaves'], :]

# 4c. Heat-map (right)
ax_h = fig.add_axes([0.35, 0.1, 0.6, 0.8])
im   = ax_h.imshow(ordered_expr, aspect='auto')
ax_h.set_yticks(range(ordered_expr.shape[0]))
ax_h.set_yticklabels(ordered_expr.index, fontsize=0)
ax_h.set_xticks(range(ordered_expr.shape[1]))
ax_h.set_xticklabels(ordered_expr.columns, rotation=90)
plt.colorbar(im, ax=ax_h, label='z-score')
plt.title('Motifs clustered by similarity')
plt.tight_layout()
plt.show()


### Clustering heatmap of designed sequences used in Figure 9 of DanioDecima manuscript.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram

# ── 0. Load & preprocess exactly as before ────────────────────────────────────
df = pd.read_csv('/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/design/analysis_25ct_20250619/comprehensive_summary_ism_90/motif_occurrence_by_celltype_combined_normalized.csv')
df = df.rename(columns={df.columns[0]: 'Gene'}).set_index('Gene')
expr = df.drop(columns='Total')

# Row-wise z-score (genes) so each gene contributes equally
expr_z = expr.sub(expr.mean(axis=1), axis=0) \
             .div(expr.std(axis=1).replace(0, 1), axis=0)

# ── 1. Cluster the *columns* (cell types) ─────────────────────────────────────
#     • We measure similarity between cell-type expression profiles (over genes)
#     • Correlation distance is common for this purpose
dist_cols   = pdist(expr_z.T, metric='correlation')         # 1 – Pearson r
linkage_cols = linkage(dist_cols, method='average')         # UPGMA

# ── 2. Plot dendrogram (top) + reordered heat-map ────────────────────────────
fig = plt.figure(figsize=(12, 9))

# 2a. Column dendrogram
ax_dc = fig.add_axes([0.30, 0.83, 0.6, 0.15])               # [l, b, w, h]
dendro_c = dendrogram(linkage_cols,
                      labels=expr_z.columns,
                      leaf_rotation=90,
                      ax=ax_dc)
ax_dc.set_ylabel('1 – corr')
ax_dc.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

# 2b. Re-order columns to match dendrogram
expr_reordered = expr_z.iloc[:, dendro_c['leaves']]

# 2c. Heat-map
ax_h = fig.add_axes([0.30, 0.10, 0.6, 0.70])
im   = ax_h.imshow(expr_reordered, aspect='auto')
ax_h.set_yticks(range(expr_reordered.shape[0]))
ax_h.set_yticklabels(expr_reordered.index, fontsize=0)
ax_h.set_xticks(range(expr_reordered.shape[1]))
ax_h.set_xticklabels(expr_reordered.columns, rotation=90)
plt.colorbar(im, ax=ax_h, label='gene z-score')
plt.title('Cell-type similarity (average-linkage)', y=1.05)
plt.tight_layout()
plt.show()


### Dendrogram of designed sequence similarites used in Figure 10 of DanioDecima manuscript.

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import scipy.spatial.distance as ssd

# -- existing Z-scored matrix (genes × cell-types) ---
# expr_z = ...

# --- build the linkage on columns (cell-types) ---
d   = ssd.pdist(expr_z.T, metric='correlation')
Z   = linkage(d, method='average')

# --- draw dendrogram only ----------------------------------------------
fig, ax = plt.subplots(figsize=(18, 5))            # wider & taller
dendro = dendrogram(
    Z,
    labels=expr_z.columns,                         # cell-type names
    leaf_rotation=90,                              # easier to read
    leaf_font_size=11,                             # bump font
    color_threshold=0.85*Z[:,2].max()               # colour big clusters
)
ax.set_ylabel('1 − corr')
ax.set_title('Cell-type similarity (average linkage)')
plt.tight_layout()
plt.show()
